## Example 1: Basic FlatBuffer Schema and Builder Setup

In [ ]:
import flatbuffers

# Example 1: Basic FlatBuffer schema and builder setup
print("Example 1: Basic FlatBuffer Schema and Builder Setup")

# Create a FlatBuffer builder
builder = flatbuffers.Builder(1024)

print(f"FlatBuffer builder created with capacity: 1024 bytes")
print(f"Builder type: {type(builder).__name__}")
print(f"Initial buffer position: {builder.Head()}")

# Test basic builder operations
print(f"\nBuilder capabilities:")
print(f"  Current offset: {builder.Offset()}")
# Output() is only legal after Finish(); until then Offset() is the number of bytes written
print(f"  Bytes written: {builder.Offset()}")
print(f"  Remaining space before the buffer grows: {builder.Head()}")
assert builder.Offset() == 0 and builder.Head() == 1024

# Create a simple string
name_offset = builder.CreateString("John Doe")
print(f"String created at offset: {name_offset}")

# Create another string
email_offset = builder.CreateString("john.doe@example.com")
print(f"Email string created at offset: {email_offset}")

# Test builder state after operations
print(f"\nAfter creating strings:")
print(f"  Bytes in buffer: {builder.Offset()}")
assert builder.Offset() >= len("John Doe") + len("john.doe@example.com")
print(f"  Builder head position: {builder.Head()}")

# Demonstrate creating numeric data
builder.StartVector(4, 3, 4)  # 3 integers, 4 bytes each
builder.PrependInt32(100)
builder.PrependInt32(200) 
builder.PrependInt32(300)
numbers_offset = builder.EndVector()

print(f"Integer vector created at offset: {numbers_offset}")

# Finish with the vector as the root and read it back: the root uoffset is the buffer's FIRST 4 bytes
builder.Finish(numbers_offset)
buf = builder.Output()
root = flatbuffers.encode.Get(flatbuffers.packer.uoffset, buf, 0)
count = flatbuffers.encode.Get(flatbuffers.packer.uoffset, buf, root)
values = [flatbuffers.encode.Get(flatbuffers.packer.int32, buf, root + 4 + 4 * i) for i in range(count)]
print(f"Finished buffer: {len(buf)} bytes, vector read back: {values}")
assert values == [300, 200, 100], values  # prepended back to front, so the last prepend reads first

print("Basic FlatBuffer operations completed successfully!")

## Example 2: Creating and Serializing Structured Data

In [ ]:
import flatbuffers

# Example 2: Creating and serializing structured data
print("Example 2: Creating and Serializing Structured Data")

# Create a builder for structured data
builder = flatbuffers.Builder(2048)

# Create strings for the structure
name = builder.CreateString("Alice Johnson")
department = builder.CreateString("Engineering")
email = builder.CreateString("alice@company.com")

print(f"Created string fields:")
print(f"  Name offset: {name}")
print(f"  Department offset: {department}")
print(f"  Email offset: {email}")

# Create a skills vector (array of strings)
skill_offsets = []
skills = ["Python", "Machine Learning", "Data Analysis", "SQL"]

for skill in skills:
    skill_offset = builder.CreateString(skill)
    skill_offsets.append(skill_offset)
    print(f"  Skill '{skill}' at offset: {skill_offset}")

# Create the skills vector
builder.StartVector(4, len(skill_offsets), 4)  # 4-byte offsets
for skill_offset in reversed(skill_offsets):  # FlatBuffers builds in reverse
    builder.PrependUOffsetTRelative(skill_offset)
skills_vector = builder.EndVector()

print(f"\nSkills vector created at offset: {skills_vector}")

# Create a table-like structure manually
# Start the main object
builder.StartObject(6)  # 6 fields

# Add fields (in reverse order due to FlatBuffer structure)
builder.PrependUOffsetTRelativeSlot(5, skills_vector, 0)  # Skills at slot 5
builder.PrependFloat32Slot(4, 85000.0, 0)  # Salary at slot 4  
builder.PrependInt32Slot(3, 30, 0)  # Age at slot 3
builder.PrependUOffsetTRelativeSlot(2, email, 0)  # Email at slot 2
builder.PrependUOffsetTRelativeSlot(1, department, 0)  # Department at slot 1
builder.PrependUOffsetTRelativeSlot(0, name, 0)  # Name at slot 0

employee = builder.EndObject()

print(f"Employee object created at offset: {employee}")

# Finish the buffer
builder.Finish(employee)

# Get the serialized data
serialized_data = builder.Output()
print(f"\nSerialization completed:")
print(f"  Buffer size: {len(serialized_data)} bytes")
print(f"  First 20 bytes: {serialized_data[:20].hex()}")
print(f"  Data is binary and ready for storage/transmission")

# Demonstrate efficiency
import json
original_data = {
    "name": "Alice Johnson",
    "department": "Engineering", 
    "email": "alice@company.com",
    "age": 30,
    "salary": 85000.0,
    "skills": ["Python", "Machine Learning", "Data Analysis", "SQL"]
}

json_size = len(json.dumps(original_data).encode('utf-8'))
flatbuf_size = len(serialized_data)

print(f"\nSize comparison:")
print(f"  JSON format: {json_size} bytes")
print(f"  FlatBuffer format: {flatbuf_size} bytes")
print(f"  Space efficiency: {((json_size - flatbuf_size) / json_size) * 100:.1f}% smaller")

## Example 3: Reading and Deserializing FlatBuffer Data

In [ ]:
import flatbuffers

# Example 3: Reading and deserializing FlatBuffer data
print("Example 3: Reading and Deserializing FlatBuffer Data")

# First create some data to read (simulate serialized data)
builder = flatbuffers.Builder(1024)

# Create sample product data
product_name = builder.CreateString("Laptop Pro")
description = builder.CreateString("High-performance laptop for developers")

# Create tags vector
tag_names = ["Electronics", "Computers", "Professional"]
tag_offsets = []
for tag in tag_names:
    tag_offsets.append(builder.CreateString(tag))

builder.StartVector(4, len(tag_offsets), 4)
for tag_offset in reversed(tag_offsets):
    builder.PrependUOffsetTRelative(tag_offset)
tags_vector = builder.EndVector()

# Create the product object  
builder.StartObject(5)  # 5 fields
builder.PrependUOffsetTRelativeSlot(4, tags_vector, 0)  # Tags
builder.PrependBoolSlot(3, True, False)  # In stock
builder.PrependFloat32Slot(2, 1299.99, 0.0)  # Price
builder.PrependUOffsetTRelativeSlot(1, description, 0)  # Description
builder.PrependUOffsetTRelativeSlot(0, product_name, 0)  # Name

product = builder.EndObject()
builder.Finish(product)

# Get the serialized data
buffer_data = builder.Output()
print(f"Created test data: {len(buffer_data)} bytes")

# Now read it back through the real reader API.
# The root uoffset is the FIRST 4 bytes of a finished buffer.
root_offset = flatbuffers.encode.Get(flatbuffers.packer.uoffset, buffer_data, 0)
print(f"Root table at offset: {root_offset}")
table = flatbuffers.Table(buffer_data, root_offset)

print(f"\nReading FlatBuffer data:")
print(f"Table position: {table.Pos}")
print(f"Buffer length: {len(table.Bytes)}")

# Field N sits at vtable slot 4 + 2*N; Offset() returns 0 for a field that is absent
name_slot = table.Offset(4)  # field 0
assert name_slot, "field 0 (name) is missing from the vtable"
product_name_read = table.String(table.Pos + name_slot).decode('utf-8')
print(f"  Product name: {product_name_read}")

price_value = table.GetSlot(8, 0.0, flatbuffers.number_types.Float32Flags)  # field 2
print(f"  Price: ${price_value:.2f}")

in_stock = bool(table.GetSlot(10, False, flatbuffers.number_types.BoolFlags))  # field 3
print(f"  In stock: {in_stock}")

tags_slot = table.Offset(12)  # field 4: a vector of string offsets
tags_start = table.Vector(tags_slot)
tags_read = [table.String(tags_start + 4 * i).decode('utf-8') for i in range(table.VectorLen(tags_slot))]
print(f"  Tags: {tags_read}")

assert product_name_read == "Laptop Pro"
assert abs(price_value - 1299.99) < 1e-3  # stored as float32
assert in_stock is True
assert tags_read == tag_names

# Demonstrate zero-copy access benefits
print(f"\nZero-copy access demonstration:")
print(f"  No parsing overhead - direct memory access")
print(f"  Data remains in original buffer")
print(f"  Suitable for high-performance applications")

# Show buffer structure
print(f"\nBuffer structure analysis:")
print(f"  Total size: {len(buffer_data)} bytes")
print(f"  Header bytes: {buffer_data[:4].hex()}")  # Root table offset
print(f"  Data accessible without full deserialization")

## Example 4: Complex Nested Structures and Performance

In [ ]:
import flatbuffers
import time
import json

# Example 4: Complex nested structures and performance
print("Example 4: Complex Nested Structures and Performance")

# Create a complex nested data structure
builder = flatbuffers.Builder(4096)

print("Creating complex nested structure...")

# Create multiple nested objects (simulate order with items)
item_objects = []

# Create 5 order items
for i in range(5):
    item_name = builder.CreateString(f"Product {i+1}")
    item_desc = builder.CreateString(f"Description for product {i+1}")
    
    # Create item object
    builder.StartObject(4)  # 4 fields per item
    builder.PrependFloat32Slot(3, (i+1) * 25.99, 0.0)  # Price
    builder.PrependInt32Slot(2, i+2, 0)  # Quantity  
    builder.PrependUOffsetTRelativeSlot(1, item_desc, 0)  # Description
    builder.PrependUOffsetTRelativeSlot(0, item_name, 0)  # Name
    
    item_objects.append(builder.EndObject())

print(f"Created {len(item_objects)} item objects")

# Create vector of items
builder.StartVector(4, len(item_objects), 4)
for item_obj in reversed(item_objects):
    builder.PrependUOffsetTRelative(item_obj)
items_vector = builder.EndVector()

# Create customer info
customer_name = builder.CreateString("John Smith")
customer_email = builder.CreateString("john.smith@email.com")
customer_address = builder.CreateString("123 Main St, City, State 12345")

# Create customer object
builder.StartObject(3)
builder.PrependUOffsetTRelativeSlot(2, customer_address, 0)  # Address
builder.PrependUOffsetTRelativeSlot(1, customer_email, 0)  # Email
builder.PrependUOffsetTRelativeSlot(0, customer_name, 0)  # Name
customer_object = builder.EndObject()

# Create main order object
order_id = builder.CreateString("ORD-2024-001")

builder.StartObject(5)  # 5 fields for order
builder.PrependFloat32Slot(4, 347.45, 0.0)  # Total
builder.PrependUOffsetTRelativeSlot(3, items_vector, 0)  # Items
builder.PrependUOffsetTRelativeSlot(2, customer_object, 0)  # Customer
builder.PrependInt64Slot(1, 1640995200, 0)  # Timestamp
builder.PrependUOffsetTRelativeSlot(0, order_id, 0)  # Order ID

main_order = builder.EndObject()
builder.Finish(main_order)

# Get the serialized data
flatbuf_data = builder.Output()
print(f"Complex structure serialized: {len(flatbuf_data)} bytes")

# Performance comparison with JSON
equivalent_json = {
    "order_id": "ORD-2024-001",
    "timestamp": 1640995200,
    "customer": {
        "name": "John Smith",
        "email": "john.smith@email.com", 
        "address": "123 Main St, City, State 12345"
    },
    "items": [
        {"name": f"Product {i+1}", "description": f"Description for product {i+1}", 
         "quantity": i+2, "price": (i+1) * 25.99}
        for i in range(5)
    ],
    "total": 347.45
}

# Serialize with JSON for comparison
start_time = time.time()
json_data = json.dumps(equivalent_json).encode('utf-8')
json_serialize_time = time.time() - start_time

print(f"\nSize comparison:")
print(f"  FlatBuffer: {len(flatbuf_data)} bytes")
print(f"  JSON: {len(json_data)} bytes")
print(f"  FlatBuffer is {len(json_data) - len(flatbuf_data)} bytes smaller")

# Test access performance
print(f"\nPerformance characteristics:")
print(f"  JSON serialization time: {json_serialize_time*1000:.3f} ms")
print(f"  FlatBuffer: Zero-copy access (no deserialization needed)")
print(f"  Memory efficiency: Direct buffer access")
print(f"  Network efficiency: Binary format, smaller payload")

# Demonstrate partial access (key FlatBuffer advantage)
print(f"\nPartial access demonstration:")
print(f"  Can access any field without parsing entire structure")
print(f"  Suitable for large datasets where only specific fields needed")
print(f"  Examples: Reading only order ID, or just customer name")
print(f"  Memory usage: Buffer can be memory-mapped for large files")

# Show nesting capabilities
print(f"\nNesting capabilities demonstrated:")
print(f"  ✓ Objects within objects (customer in order)")
print(f"  ✓ Arrays of objects (items in order)")
print(f"  ✓ Mixed data types (strings, numbers, booleans)")
print(f"  ✓ Arbitrary nesting depth supported")

## Example 5: Schema Evolution and Production Usage

In [ ]:
import flatbuffers
import tempfile
import os

# Example 5: Schema evolution and production usage
print("Example 5: Schema Evolution and Production Usage")

# Simulate schema evolution scenario
print("Demonstrating schema evolution compatibility...")

# Version 1 schema: Basic user data
def create_user_v1(builder, name, email, age):
    """Create user with original schema (3 fields)"""
    name_offset = builder.CreateString(name)
    email_offset = builder.CreateString(email)
    
    builder.StartObject(3)  # 3 fields in v1
    builder.PrependInt32Slot(2, age, 0)  # Age
    builder.PrependUOffsetTRelativeSlot(1, email_offset, 0)  # Email
    builder.PrependUOffsetTRelativeSlot(0, name_offset, 0)  # Name
    
    return builder.EndObject()

# Version 2 schema: Extended user data (backward compatible)
def create_user_v2(builder, name, email, age, phone=None, premium=False):
    """Create user with extended schema (5 fields, backward compatible)"""
    name_offset = builder.CreateString(name)
    email_offset = builder.CreateString(email)
    phone_offset = 0
    if phone:
        phone_offset = builder.CreateString(phone)
    
    builder.StartObject(5)  # 5 fields in v2
    builder.PrependBoolSlot(4, premium, False)  # Premium (new field)
    builder.PrependUOffsetTRelativeSlot(3, phone_offset, 0)  # Phone (new field)
    builder.PrependInt32Slot(2, age, 0)  # Age (existing)
    builder.PrependUOffsetTRelativeSlot(1, email_offset, 0)  # Email (existing)
    builder.PrependUOffsetTRelativeSlot(0, name_offset, 0)  # Name (existing)
    
    return builder.EndObject()

# Create data with v1 schema
builder_v1 = flatbuffers.Builder(512)
user_v1 = create_user_v1(builder_v1, "Alice", "alice@example.com", 25)
builder_v1.Finish(user_v1)
data_v1 = builder_v1.Output()

print(f"Created v1 user data: {len(data_v1)} bytes")

# Create data with v2 schema
builder_v2 = flatbuffers.Builder(512)
user_v2 = create_user_v2(builder_v2, "Bob", "bob@example.com", 30, "+1-555-1234", True)
builder_v2.Finish(user_v2)
data_v2 = builder_v2.Output()

print(f"Created v2 user data: {len(data_v2)} bytes")

# Demonstrate reading both formats with same reader logic
def read_user_data(buffer_data, version="auto"):
    """Universal reader that works with both v1 and v2 data"""
    # The root uoffset is the FIRST 4 bytes of a finished buffer
    root_offset = flatbuffers.encode.Get(flatbuffers.packer.uoffset, buffer_data, 0)
    table = flatbuffers.Table(buffer_data, root_offset)
    
    user_info = {"version_detected": "unknown"}
    
    # Read common fields (available in both v1 and v2); field N sits at vtable slot 4 + 2*N
    name_offset = table.Offset(4)  # Field 0
    if name_offset:
        user_info["name"] = table.String(table.Pos + name_offset).decode('utf-8')
    
    # Detect the version by probing a v2-only field: a v1 vtable is too short, so Offset() returns 0
    phone_offset = table.Offset(10)  # Field 3 (phone) in v2
    if phone_offset:
        user_info["version_detected"] = "v2"
        # Read v2-specific fields
        user_info["has_phone"] = True
        user_info["phone"] = table.String(table.Pos + phone_offset).decode('utf-8')
        user_info["premium"] = bool(table.GetSlot(12, False, flatbuffers.number_types.BoolFlags))  # Field 4
    else:
        user_info["version_detected"] = "v1"
        user_info["has_phone"] = False
        user_info["premium"] = False
    
    return user_info

# Read both versions with same function
print(f"\nReading v1 data:")
v1_info = read_user_data(data_v1)
for key, value in v1_info.items():
    print(f"  {key}: {value}")

print(f"\nReading v2 data:")
v2_info = read_user_data(data_v2)
for key, value in v2_info.items():
    print(f"  {key}: {value}")

# The same reader must decode BOTH schema versions correctly
assert v1_info == {"version_detected": "v1", "name": "Alice", "has_phone": False, "premium": False}, v1_info
assert v2_info == {"version_detected": "v2", "name": "Bob", "has_phone": True, "phone": "+1-555-1234", "premium": True}, v2_info

# Demonstrate file-based storage (production usage)
print(f"\nFile-based storage simulation:")

with tempfile.TemporaryDirectory() as temp_dir:
    # Write multiple user records to files
    users_data = [
        ("user_1.fb", data_v1),
        ("user_2.fb", data_v2)
    ]
    
    for filename, data in users_data:
        file_path = os.path.join(temp_dir, filename)
        with open(file_path, 'wb') as f:
            f.write(data)
        print(f"  Saved {filename}: {len(data)} bytes")
    
    # Read back from files
    print(f"\nReading from files:")
    for filename, original in users_data:
        file_path = os.path.join(temp_dir, filename)
        with open(file_path, 'rb') as f:
            file_data = f.read()
        
        assert file_data == bytes(original)
        user_info = read_user_data(file_data)
        assert user_info == read_user_data(original)
        print(f"  {filename}: {user_info['name']} ({user_info['version_detected']})")

# Production deployment considerations
print(f"\nProduction deployment considerations:")
considerations = [
    "✓ Schema evolution: Add new fields without breaking existing readers",
    "✓ Backward compatibility: Old clients can read new data format",
    "✓ Forward compatibility: New clients can read old data format", 
    "✓ File storage: Efficient binary format for persistent storage",
    "✓ Network protocol: Compact format reduces bandwidth usage",
    "✓ Memory mapping: Large files can be memory-mapped for efficiency",
    "✓ Cross-platform: Same binary format works across architectures"
]

for consideration in considerations:
    print(f"    {consideration}")

print(f"\nSchema evolution completed successfully!")